In [11]:
"""
MLP MNIST 训练脚本
"""
import os
import argparse
import urllib.request
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


class MLP(nn.Module):
    def __init__(self, input_size=784, hidden1=256, hidden2=128, num_classes=10, dropout=0.2):
        super(MLP, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(input_size, hidden1)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.fc3 = nn.Linear(hidden2, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        return x


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def get_args():
    parser = argparse.ArgumentParser(description='MLP MNIST Training')
    parser.add_argument('--epochs', type=int, default=10)
    parser.add_argument('--batch-size', type=int, default=64)
    parser.add_argument('--lr', type=float, default=0.001)
    parser.add_argument('--hidden1', type=int, default=256)
    parser.add_argument('--hidden2', type=int, default=128)
    parser.add_argument('--dropout', type=float, default=0.2)
    parser.add_argument('--data-dir', type=str, default='/home/lf/data')
    parser.add_argument('--save-dir', type=str, default='/home/lf/checkpoints')
    parser.add_argument('--seed', type=int, default=42)
    return parser.parse_known_args()[0]


def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_data_loaders(data_dir, batch_size):
    raw_dir = os.path.join(data_dir, 'MNIST', 'raw')
    os.makedirs(raw_dir, exist_ok=True)

    files = [
        'train-images-idx3-ubyte.gz',
        'train-labels-idx1-ubyte.gz',
        't10k-images-idx3-ubyte.gz',
        't10k-labels-idx1-ubyte.gz',
    ]

    # 检查 raw 文件是否齐全，不齐全才尝试国内镜像下载
    all_exist = all(os.path.exists(os.path.join(raw_dir, f)) for f in files)
    if not all_exist:
        mirrors = [
            'https://mirror.coggle.club/mnist/',
            'https://ghproxy.com/https://raw.githubusercontent.com/fgnt/mnist/master/',
            'https://ossci-datasets.s3.amazonaws.com/mnist/',
        ]
        for fname in files:
            fpath = os.path.join(raw_dir, fname)
            if os.path.exists(fpath):
                continue
            for mirror in mirrors:
                try:
                    print(f'尝试从 {mirror} 下载 {fname} ...')
                    urllib.request.urlretrieve(mirror + fname, fpath)
                    print(f'{fname} 下载成功')
                    break
                except Exception as e:
                    print(f'失败: {e}')
                    continue

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    # download=True: raw文件已存在则不会重新下载，只会解压处理成processed文件
    train_dataset = datasets.MNIST(root=data_dir, train=True, download=True, transform=transform)
    test_dataset = datasets.MNIST(root=data_dir, train=False, download=True, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    return train_loader, test_loader


def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        loss = criterion(output, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * data.size(0)
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += target.size(0)
    return running_loss / total, 100.0 * correct / total


@torch.no_grad()
def evaluate(model, test_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        loss = criterion(output, target)
        running_loss += loss.item() * data.size(0)
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += target.size(0)
    return running_loss / total, 100.0 * correct / total


def plot_curves(train_losses, test_losses, train_accs, test_accs, save_path):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    epochs = range(1, len(train_losses) + 1)
    ax1.plot(epochs, train_losses, 'b-', label='Train Loss')
    ax1.plot(epochs, test_losses, 'r-', label='Test Loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Training and Test Loss')
    ax1.legend(); ax1.grid(True, alpha=0.3)
    ax2.plot(epochs, train_accs, 'b-', label='Train Accuracy')
    ax2.plot(epochs, test_accs, 'r-', label='Test Accuracy')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)'); ax2.set_title('Training and Test Accuracy')
    ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path, dpi=150); plt.close()
    print(f"曲线已保存: {save_path}")


def main():
    args = get_args()
    set_seed(args.seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    train_loader, test_loader = get_data_loaders(args.data_dir, args.batch_size)
    print(f"训练集: {len(train_loader.dataset)} 张, 测试集: {len(test_loader.dataset)} 张")

    model = MLP(hidden1=args.hidden1, hidden2=args.hidden2, dropout=args.dropout).to(device)
    print(f"模型参数量: {count_parameters(model):,}")

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=args.lr)

    os.makedirs(args.save_dir, exist_ok=True)
    train_losses, test_losses, train_accs, test_accs = [], [], [], []
    best_acc = 0.0

    print(f"\n开始训练，共 {args.epochs} 个 epoch...")
    print("-" * 70)
    for epoch in range(1, args.epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)
        train_losses.append(train_loss); test_losses.append(test_loss)
        train_accs.append(train_acc); test_accs.append(test_acc)
        print(f"Epoch {epoch:2d}/{args.epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | "
              f"Test Loss: {test_loss:.4f} Acc: {test_acc:.2f}%")
        if test_acc > best_acc:
            best_acc = test_acc
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'test_acc': test_acc, 'args': vars(args)},
                       os.path.join(args.save_dir, 'mlp_mnist_best.pth'))

    print("-" * 70)
    print(f"训练完成! 最佳测试准确率: {best_acc:.2f}%")

    torch.save({'epoch': args.epochs, 'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'test_acc': test_acc, 'args': vars(args)},
               os.path.join(args.save_dir, 'mlp_mnist_last.pth'))

    plot_curves(train_losses, test_losses, train_accs, test_accs,
                os.path.join(args.save_dir, 'training_curves.png'))

    with open(os.path.join(args.save_dir, 'training_log.txt'), 'w') as f:
        f.write("MLP MNIST Training Log\n" + "=" * 50 + "\n")
        f.write(f"参数量: {count_parameters(model):,}\n")
        f.write(f"最佳测试准确率: {best_acc:.2f}%\n")
        f.write(f"超参数: {vars(args)}\n\n")
        f.write("Epoch,Train_Loss,Train_Acc,Test_Loss,Test_Acc\n")
        for i in range(len(train_losses)):
            f.write(f"{i+1},{train_losses[i]:.4f},{train_accs[i]:.2f},{test_losses[i]:.4f},{test_accs[i]:.2f}\n")


if __name__ == "__main__":
    main()



使用设备: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Using downloaded and verified file: /home/lf/data/MNIST/raw/train-images-idx3-ubyte.gz
Extracting /home/lf/data/MNIST/raw/train-images-idx3-ubyte.gz to /home/lf/data/MNIST/raw

Using downloaded and verified file: /home/lf/data/MNIST/raw/train-labels-idx1-ubyte.gz
Extracting /home/lf/data/MNIST/raw/train-labels-idx1-ubyte.gz to /home/lf/data/MNIST/raw

Using downloaded and verified file: /home/lf/data/MNIST/raw/t10k-images-idx3-ubyte.gz
Extracting /home/lf/data/MNIST/raw/t10k-images-idx3-ubyte.gz to /home/lf/data/MNIST/raw

Using downloaded and verified file: /home/lf/data/MNIST/raw/t10k-labels-idx1-ubyte.gz
Extracting /home/lf/data/MNIST/raw/t10k-labels-idx1-ubyte.gz to /home/lf/data/MNIST/raw

训练集: 60000 张, 测试集: 10000 张
模型参数量: 235,146

开始训练，共 10 个 epoch...
----------------------------------------------------------------------
Epoch  1/10 | Train Loss: 0.2791 Acc: 91.49% | Test Loss: 0.1235 Acc: 96.19%
Epoch  2/10 | Train Loss: 0.1305 

In [12]:
"""
MLP MNIST 推理测试
用训练好的模型预测手写数字
"""
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备: {device}')


# ===== 模型结构（必须和训练时完全一致）=====
class MLP(nn.Module):
    def __init__(self, input_size=784, hidden1=256, hidden2=128, num_classes=10, dropout=0.2):
        super(MLP, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(input_size, hidden1)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.fc3 = nn.Linear(hidden2, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        return x


# ===== 加载训练好的模型 =====
model = MLP().to(device)
checkpoint = torch.load('/home/lf/checkpoints/mlp_mnist_best.pth', map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'模型加载成功！训练时测试准确率: {checkpoint["test_acc"]:.2f}%')


# ===== 方式A：从测试集随机取一张预测 =====
print('\n===== 方式A：从测试集随机取一张 =====')
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
test_dataset = datasets.MNIST(root='/home/lf/data', train=False, download=True, transform=transform)

idx = np.random.randint(len(test_dataset))
image, true_label = test_dataset[idx]

with torch.no_grad():
    output = model(image.unsqueeze(0).to(device))
    probs = torch.softmax(output, dim=1).squeeze(0).cpu().numpy()
    pred = output.argmax(1).item()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
display_img = image.squeeze() * 0.3081 + 0.1307
ax1.imshow(display_img, cmap='gray')
ax1.set_title(f'预测: {pred}  真实: {true_label}', fontsize=14)
ax1.axis('off')

digits = range(10)
colors = ['green' if i == pred else 'steelblue' for i in digits]
ax2.bar(digits, probs, color=colors)
ax2.set_xlabel('数字'); ax2.set_ylabel('概率'); ax2.set_title('各类别概率')
ax2.set_xticks(digits); ax2.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('/home/lf/inference_result.png', dpi=150)
plt.show()

print(f'预测结果: {pred}')
print(f'真实标签: {true_label}')
print(f'预测{"正确" if pred == true_label else "错误"}')
print('结果图已保存: /home/lf/inference_result.png')


# ===== 方式B：用你自己的手写数字图片预测 =====
# 把你的图片放到 /home/lf/ 下面，然后把下面的文件名改成你的图片名
# 注意：图片要是黑底白字（数字白色，背景黑色），和MNIST格式一致
print('\n===== 方式B：用自己的图片预测（取消注释才能用）=====')
"""
image_path = '/home/lf/你的图片.png'  # 改成你的图片路径

img = Image.open(image_path).convert('L')
img = img.resize((28, 28), Image.LANCZOS)
tensor = transform(img)

with torch.no_grad():
    output = model(tensor.unsqueeze(0).to(device))
    probs = torch.softmax(output, dim=1).squeeze(0).cpu().numpy()
    pred = output.argmax(1).item()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(img, cmap='gray')
ax1.set_title(f'预测结果: {pred}', fontsize=16)
ax1.axis('off')
ax2.bar(range(10), probs, color=['green' if i == pred else 'steelblue' for i in range(10)])
ax2.set_xlabel('数字'); ax2.set_ylabel('概率'); ax2.set_xticks(range(10)); ax2.set_ylim(0, 1)
plt.tight_layout(); plt.show()
print(f'模型预测: 数字 {pred}')
"""


使用设备: cuda
模型加载成功！训练时测试准确率: 98.06%

===== 方式A：从测试集随机取一张 =====


/tmp/ipykernel_29677/2521318764.py:77: UserWarning: Glyph 39044 (\N{CJK UNIFIED IDEOGRAPH-9884}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_29677/2521318764.py:77: UserWarning: Glyph 27979 (\N{CJK UNIFIED IDEOGRAPH-6D4B}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_29677/2521318764.py:77: UserWarning: Glyph 30495 (\N{CJK UNIFIED IDEOGRAPH-771F}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_29677/2521318764.py:77: UserWarning: Glyph 23454 (\N{CJK UNIFIED IDEOGRAPH-5B9E}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_29677/2521318764.py:77: UserWarning: Glyph 25968 (\N{CJK UNIFIED IDEOGRAPH-6570}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_29677/2521318764.py:77: UserWarning: Glyph 23383 (\N{CJK UNIFIED IDEOGRAPH-5B57}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_29677/2521318764.py:77: UserWarning: Glyph 27010 (\N{CJK UNIFIED IDEOGRAP

预测结果: 6
真实标签: 6
预测正确
结果图已保存: /home/lf/inference_result.png

===== 方式B：用自己的图片预测（取消注释才能用）=====


/tmp/ipykernel_29677/2521318764.py:79: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


"\nimage_path = '/home/lf/你的图片.png'  # 改成你的图片路径\n\nimg = Image.open(image_path).convert('L')\nimg = img.resize((28, 28), Image.LANCZOS)\ntensor = transform(img)\n\nwith torch.no_grad():\n    output = model(tensor.unsqueeze(0).to(device))\n    probs = torch.softmax(output, dim=1).squeeze(0).cpu().numpy()\n    pred = output.argmax(1).item()\n\nfig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))\nax1.imshow(img, cmap='gray')\nax1.set_title(f'预测结果: {pred}', fontsize=16)\nax1.axis('off')\nax2.bar(range(10), probs, color=['green' if i == pred else 'steelblue' for i in range(10)])\nax2.set_xlabel('数字'); ax2.set_ylabel('概率'); ax2.set_xticks(range(10)); ax2.set_ylim(0, 1)\nplt.tight_layout(); plt.show()\nprint(f'模型预测: 数字 {pred}')\n"

In [15]:
import torch, torch.nn as nn, numpy as np
import torch
import torch.nn as nn
import numpy as np
from PIL import Image, ImageOps
from torchvision import transforms
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class MLP(nn.Module):
    def __init__(self, input_size=784, hidden1=256, hidden2=128, num_classes=10, dropout=0.2):
        super(MLP, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(input_size, hidden1)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.fc3 = nn.Linear(hidden2, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

model = MLP().to(device)
ckpt = torch.load('/home/lf/checkpoints/mlp_mnist_best.pth', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print('模型加载成功')

img = Image.open('你的图片.png').convert('L').resize((28, 28), Image.LANCZOS)
if np.array(img).mean() > 128:
    img = ImageOps.invert(img)
    print('白底黑字，已自动反色')

t = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])(img).unsqueeze(0).to(device)

with torch.no_grad():
    out = model(t)
    pred = out.argmax(1).item()
    prob = torch.softmax(out, 1).max().item()

print(f'预测结果: 数字 {pred}，置信度 {prob:.1%}')
plt.imshow(img, cmap='gray')
plt.title(f'预测: {pred}')
plt.axis('off')
plt.show()



模型加载成功
白底黑字，已自动反色
预测结果: 数字 7，置信度 65.0%


/tmp/ipykernel_29677/329897361.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
